In [1]:
# Cell 1: dựng lại bộ công cụ cho notebook mới
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")  # kết nối Python với BigQuery

def mb(n):
    # đổi byte sang MB; n là None (job chưa xong) thì trả về None, không ép thành 0
    return None if n is None else round(n / 1024 / 1024, 2)

def check(sql):
    # dry run: chỉ ước lượng, không chạy thật
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)  # chỉ tạo cấu hình
    job = client.query(sql, job_config=cfg)                              # ước lượng xảy ra ở dòng này
    print(f"Sẽ quét: {mb(job.total_bytes_processed)} MB")

def stats(sql, max_mb=100):
    # chạy thật, tắt cache, giới hạn bytes billed, in processed và billed
    cfg = bigquery.QueryJobConfig(use_query_cache=False,
                                  maximum_bytes_billed=max_mb * 1024 * 1024)
    job = client.query(sql, job_config=cfg)
    df = job.result().to_dataframe()   # job.result() chờ job xong rồi mới đọc số byte
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df

print("Đã sẵn sàng")

Đã sẵn sàng


In [2]:
# Cell 2: xem ai có quyền gì trên dataset lab (không in email vì repo công khai)
ds = client.get_dataset(f"{PROJECT}.lab")      # lấy thông tin dataset

for entry in ds.access_entries:                # lặp qua từng dòng quyền, mỗi vòng gán một dòng vào entry
    # role: quyền; entity_type: loại đối tượng (người, nhóm đặc biệt...)
    # cố ý KHÔNG in entry.entity_id vì có thể chứa email
    print(entry.role, "|", entry.entity_type)

WRITER | specialGroup
OWNER | specialGroup
OWNER | userByEmail
READER | specialGroup


In [3]:
# Cell 3: xem tên cột và kiểu dữ liệu của bảng fact_items_plain
# get_table chỉ đọc thông tin bảng, không chạy query nên không tốn byte
t = client.get_table(f"{PROJECT}.lab.fact_items_plain")   # lấy thông tin bảng, gán vào t

for field in t.schema:                                    # mỗi vòng gán một cột vào field
    print(field.name, "|", field.field_type)              # in tên cột và kiểu dữ liệu

print("Số dòng:", t.num_rows)                             # số dòng của bảng

order_item_id | INTEGER
order_id | INTEGER
user_id | INTEGER
product_id | INTEGER
order_date_key | INTEGER
status | STRING
sale_price | FLOAT
cost | FLOAT
gross_profit | FLOAT
order_date | DATE
Số dòng: 180771


In [6]:
# Cell 4: tạo lại view v_net_sales với quy tắc mới (chỉ đơn Complete)
# View không lưu dữ liệu, chỉ lưu câu SELECT; ghi đè định nghĩa cũ là an toàn
ddl = f"""
CREATE OR REPLACE VIEW `{PROJECT}.lab.v_net_sales` AS
SELECT
  order_item_id,
  order_date,
  product_id,
  status,
  sale_price,
  gross_profit
FROM `{PROJECT}.lab.fact_items_plain`
WHERE status = 'Complete'   -- doanh thu thực tế chỉ tính đơn đã hoàn tất
"""

client.query(ddl).result()   # gửi lệnh tạo view; result() chờ xong rồi mới chạy tiếp
print("Đã tạo lại view")

Đã tạo lại view


In [5]:
# Cell 4b: kiểm tra chính xác các giá trị status đang có trong bảng
sql_status = f"""
SELECT status, COUNT(*) AS so_dong
FROM `{PROJECT}.lab.fact_items_plain`
GROUP BY status
ORDER BY so_dong DESC
"""

check(sql_status)              # dry run trước, thấy nhỏ rồi mới chạy thật
df_status = stats(sql_status)  # chạy thật, in processed và billed
print(df_status)

Sẽ quét: 1.77 MB
processed: 1.77 MB | billed: 10.0 MB
       status  so_dong
0     Shipped    53996
1    Complete    45282
2  Processing    36063
3   Cancelled    27373
4    Returned    18057


In [7]:
# Cell 5: dry run truy vấn qua view (không chạy thật, không tốn byte)
sql_view = f"""
SELECT COUNT(*) AS so_dong, SUM(sale_price) AS net_revenue
FROM `{PROJECT}.lab.v_net_sales`
"""
check(sql_view)   # in số MB sẽ quét

Sẽ quét: 3.15 MB


In [8]:
# Cell 6: chạy thật qua view và trực tiếp trên bảng gốc, so sánh kết quả và byte
sql_direct = f"""
SELECT COUNT(*) AS so_dong, SUM(sale_price) AS net_revenue
FROM `{PROJECT}.lab.fact_items_plain`
WHERE status = 'Complete'
"""

print("Qua view:")
df_view = stats(sql_view)        # stats in processed và billed, trả về DataFrame

print("Truy vấn trực tiếp bảng gốc:")
df_direct = stats(sql_direct)

print(df_view)
print(df_direct)                 # hai bảng kết quả phải giống hệt nhau

Qua view:
processed: 3.15 MB | billed: 10.0 MB
Truy vấn trực tiếp bảng gốc:
processed: 3.15 MB | billed: 10.0 MB
   so_dong   net_revenue
0    45282  2.702568e+06
   so_dong   net_revenue
0    45282  2.702568e+06


In [9]:
# Cell 7: tạo materialized view doanh thu theo ngày (chỉ đơn Complete)
# MV lưu sẵn kết quả GROUP BY, nên mỗi ngày chỉ còn một dòng
ddl_mv = f"""
CREATE MATERIALIZED VIEW `{PROJECT}.lab.mv_net_sales_daily` AS
SELECT
  order_date,
  COUNT(*)          AS so_dong,
  SUM(sale_price)   AS net_revenue,
  SUM(gross_profit) AS gross_profit
FROM `{PROJECT}.lab.fact_items_plain`
WHERE status = 'Complete'      -- cùng quy tắc doanh thu thực tế như view
GROUP BY order_date
"""

client.query(ddl_mv).result()   # result() chờ xong rồi mới chạy tiếp
print("Đã tạo materialized view")

Đã tạo materialized view


In [10]:
# Cell 8: hỏi cùng một câu (tổng doanh thu) qua MV, so với 3,15 MB của bảng gốc
sql_mv = f"""
SELECT SUM(so_dong) AS so_dong, SUM(net_revenue) AS net_revenue
FROM `{PROJECT}.lab.mv_net_sales_daily`
"""

check(sql_mv)            # dry run trước
df_mv = stats(sql_mv)    # chạy thật, in processed và billed
print(df_mv)

Sẽ quét: 0.06 MB
processed: 0.06 MB | billed: 10.0 MB
   so_dong   net_revenue
0    45282  2.702568e+06


In [13]:
# Cell 9: kiểm tra gross_profit và biên lợi nhuận bằng SQL trên view v_net_sales
sql_gp = f"""
SELECT
  SUM(sale_price)   AS net_revenue,
  SUM(gross_profit) AS gross_profit,
  SUM(gross_profit) / SUM(sale_price) AS margin   -- tổng chia tổng
FROM `{PROJECT}.lab.v_net_sales`
"""

check(sql_gp)             # dry run trước
df_gp = stats(sql_gp)     # chạy thật, in processed và billed
print(df_gp)

Sẽ quét: 0.04 MB
processed: 0.04 MB | billed: 10.0 MB
    net_revenue  gross_profit    margin
0  2.702568e+06  1.402617e+06  0.518994


In [14]:
# Cell 10: kiểm tra độc lập gross_profit trên bảng dwh.fact_order_items (MV không dùng bảng này)
sql_dwh = f"""
SELECT
  COUNT(*)          AS so_dong,
  SUM(sale_price)   AS net_revenue,
  SUM(gross_profit) AS gross_profit,
  SUM(gross_profit) / SUM(sale_price) AS margin
FROM `{PROJECT}.dwh.fact_order_items`
WHERE status = 'Complete'     -- cùng quy tắc doanh thu thực tế
"""

check(sql_dwh)                # dry run trước
df_dwh = stats(sql_dwh)       # chạy thật
print(df_dwh)

Sẽ quét: 4.53 MB
processed: 4.53 MB | billed: 10.0 MB
   so_dong   net_revenue  gross_profit    margin
0    45282  2.702568e+06  1.402617e+06  0.518994


In [15]:
# Cell 11: doanh thu theo tháng của 6 tháng gần nhất, kèm số ngày có dữ liệu trong tháng
sql_month = f"""
SELECT
  FORMAT_DATE('%Y-%m', order_date) AS thang,     -- %Y là năm 4 chữ số (viết hoa)
  COUNT(*)                          AS so_ngay,   -- mỗi dòng của MV là một ngày
  SUM(so_dong)                      AS so_dong,
  SUM(net_revenue)                  AS net_revenue
FROM `{PROJECT}.lab.mv_net_sales_daily`
GROUP BY thang
ORDER BY thang DESC
LIMIT 6
"""

check(sql_month)               # dry run trước
df_month = stats(sql_month)    # chạy thật
print(df_month)

Sẽ quét: 0.06 MB
processed: 0.06 MB | billed: 10.0 MB
     thang  so_ngay  so_dong    net_revenue
0  2026-10        3      291   17677.659968
1  2026-09       30     3447  201673.830197
2  2026-08       31     2127  126907.110205
3  2026-07       31     1761  105777.310106
4  2026-06       30     1545   91669.990093
5  2026-05       31     1428   87024.090136


In [16]:
# Cell 12: các query gần đây trong project, tìm những query do Data Studio gửi
sql_jobs = f"""
SELECT
  creation_time,
  total_bytes_processed,
  total_bytes_billed,
  LEFT(query, 150) AS query_dau            -- chỉ lấy 150 ký tự đầu của câu SQL
FROM `{PROJECT}.region-us.INFORMATION_SCHEMA.JOBS`
WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)  -- 24 giờ để tránh bảng rỗng
ORDER BY creation_time DESC
LIMIT 20
"""

check(sql_jobs)
df_jobs = stats(sql_jobs)
print(df_jobs)

Sẽ quét: 0.35 MB
processed: 0.06 MB | billed: 20.0 MB
                      creation_time  total_bytes_processed  \
0  2026-10-06 06:11:45.750000+00:00                  63696   
1  2026-10-06 06:03:34.722000+00:00                  42464   
2  2026-10-06 05:58:33.878000+00:00                  63696   
3  2026-10-06 05:58:33.784000+00:00                  42464   
4  2026-10-06 05:58:33.780000+00:00                  42464   
5  2026-10-06 05:58:33.778000+00:00                  42464   
6  2026-10-06 05:57:55.657000+00:00                      0   
7  2026-10-06 05:57:55.649000+00:00                      0   
8  2026-10-06 05:57:55.647000+00:00                      0   
9  2026-10-06 05:57:55.646000+00:00                      0   
10 2026-10-06 05:57:39.902000+00:00                  42464   
11 2026-10-06 05:57:39.896000+00:00                  42464   
12 2026-10-06 05:57:39.889000+00:00                  42464   
13 2026-10-06 05:57:39.060000+00:00                  63696   
14 2026-10-06 05

In [17]:
# Cell 13: tổng trong khoảng 2025-01-15 đến 2026-10-06 (cùng khoảng với Date range control)
sql_range = f"""
SELECT
  SUM(so_dong)       AS so_dong,
  SUM(net_revenue)   AS net_revenue,
  SUM(gross_profit)  AS gross_profit,
  SUM(gross_profit) / SUM(net_revenue) AS margin   -- tổng chia tổng
FROM `{PROJECT}.lab.mv_net_sales_daily`
WHERE order_date BETWEEN '2025-01-15' AND '2026-10-06'
"""

check(sql_range)             # dry run trước
df_range = stats(sql_range)  # chạy thật
print(df_range)

Sẽ quét: 0.08 MB
processed: 0.08 MB | billed: 10.0 MB
   so_dong   net_revenue   gross_profit    margin
0    25658  1.529873e+06  793320.823706  0.518553


In [29]:
# Cell 12b: query liên quan đến MV, kèm thời gian chạy và có dùng cache hay không
# Giờ trong JOBS là UTC (giờ Việt Nam = UTC + 7); cố ý KHÔNG chọn user_email vì repo công khai
sql_jobs2 = f"""
SELECT
  creation_time,
  TIMESTAMP_DIFF(end_time, start_time, MILLISECOND) AS ms,   -- thời gian chạy trong BigQuery
  cache_hit,                                                  -- TRUE nếu kết quả lấy từ cache
  total_bytes_processed,
  total_bytes_billed,
  LEFT(query, 60) AS query_dau
FROM `{PROJECT}.region-us.INFORMATION_SCHEMA.JOBS`
WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
  AND query LIKE '%mv_net_sales_daily%'
  AND query NOT LIKE '%INFORMATION_SCHEMA%'                   -- loại chính câu truy vấn này
ORDER BY creation_time DESC
LIMIT 30
"""

check(sql_jobs2)               # dry run trước
df_jobs2 = stats(sql_jobs2)    # chạy thật
print(df_jobs2)

Sẽ quét: 0.54 MB
processed: 0.1 MB | billed: 20.0 MB
                      creation_time   ms  cache_hit  total_bytes_processed  \
0  2026-10-06 06:21:39.963000+00:00  222      False                  63696   
1  2026-10-06 06:21:33.593000+00:00  230      False                  42464   
2  2026-10-06 06:21:24.422000+00:00  255      False                  42464   
3  2026-10-06 06:21:17.275000+00:00  203      False                  42464   
4  2026-10-06 06:17:21.887000+00:00  252      False                  42464   
5  2026-10-06 06:12:18.486000+00:00  227      False                  84928   
6  2026-10-06 06:11:45.750000+00:00  477      False                  63696   
7  2026-10-06 06:03:34.722000+00:00  378      False                  42464   
8  2026-10-06 05:58:33.878000+00:00  212      False                  63696   
9  2026-10-06 05:58:33.784000+00:00  271      False                  42464   
10 2026-10-06 05:58:33.780000+00:00  220      False                  42464   
11 2026-10-

In [31]:
# Cell 14: in đầy đủ các query gần nhất liên quan đến MV (thẻ KPI và biểu đồ của Data Studio)
sql_q = f"""
SELECT creation_time, query
FROM `{PROJECT}.region-us.INFORMATION_SCHEMA.JOBS`
WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
  AND query LIKE '%mv_net_sales_daily%'
  AND query NOT LIKE '%INFORMATION_SCHEMA%'      -- loại chính câu truy vấn này
ORDER BY creation_time DESC
LIMIT 8
"""

check(sql_q)                  # dry run trước
df_q = stats(sql_q)           # chạy thật

for _, row in df_q.iterrows():            # mỗi vòng gán một dòng kết quả vào row
    print(row["creation_time"])           # giờ UTC của query
    print(row["query"])                   # toàn bộ câu SQL, tìm phần WHERE
    print("-" * 60)                       # đường kẻ cho dễ đọc

Sẽ quét: 0.54 MB
processed: 0.09 MB | billed: 20.0 MB
2026-10-06 06:21:39.963000+00:00
SELECT SUM(clmn1_) AS `t0_D12196881872859194587`, SUM(clmn0_) AS `t0_D17119531564286947524` FROM (
SELECT t0.`gross_profit` AS clmn0_, t0.`net_revenue` AS clmn1_, t0.`order_date` AS clmn2_ FROM `bq-learning-510104.lab.mv_net_sales_daily` AS t0
) WHERE (clmn2_ >= DATE '2025-01-15' AND clmn2_ <= DATE '2026-10-06') LIMIT 2000001;
------------------------------------------------------------
2026-10-06 06:21:33.593000+00:00
SELECT SUM(clmn0_) AS `t0_qt_1bwl9na37d` FROM (
SELECT t0.`gross_profit` AS clmn0_, t0.`order_date` AS clmn1_ FROM `bq-learning-510104.lab.mv_net_sales_daily` AS t0
) WHERE (clmn1_ >= DATE '2025-01-15' AND clmn1_ <= DATE '2026-10-06') LIMIT 2000001;
------------------------------------------------------------
2026-10-06 06:21:24.422000+00:00
SELECT SUM(clmn1_) AS `t0_qt_kfsmsha37d` FROM (
SELECT t0.`order_date` AS clmn0_, t0.`so_dong` AS clmn1_ FROM `bq-learning-510104.lab.mv_net_sales